<p estilo="text-align:centro">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Logotipo de Skills Network">
    </a>
</p>


# **Análisis de ubicaciones de sitios de lanzamiento con Folium**


Tiempo estimado necesario: **40** minutos


La tasa de éxito del lanzamiento puede depender de muchos factores, como la masa de la carga útil, el tipo de órbita, etc. También puede depender de la ubicación y las proximidades del lugar de lanzamiento, es decir, la posición inicial de las trayectorias de los cohetes. Encontrar una ubicación óptima para construir un sitio de lanzamiento ciertamente implica muchos factores y, con suerte, podríamos descubrir algunos de los factores analizando las ubicaciones de los sitios de lanzamiento existentes.


En los laboratorios de análisis de datos exploratorios anteriores, visualizó el conjunto de datos del lanzamiento de SpaceX utilizando "matplotlib" y "seaborn" y descubrió algunas correlaciones preliminares entre el sitio de lanzamiento y las tasas de éxito. En esta práctica de laboratorio, realizará análisis visuales más interactivos utilizando "Folium".


## Objetivos


Este laboratorio contiene las siguientes tareas:
- **TAREA 1:** Marcar todos los sitios de lanzamiento en un mapa
- **TAREA 2:** Marque los lanzamientos exitosos/fallidos para cada sitio en el mapa
- **TAREA 3:** Calcular las distancias entre un sitio de lanzamiento y sus proximidades.

Después de completar las tareas anteriores, debería poder encontrar algunos patrones geográficos sobre los sitios de lanzamiento.


Primero importemos los paquetes de Python necesarios para esta práctica de laboratorio:


In [ ]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

In [8]:
import folium
import wget
import pandas as pd

In [3]:
# Importar el complemento MarkerCluster de folio
from folium.plugins import MarkerCluster
# Importar complemento Folium MousePosition
from folium.plugins import MousePosition
# Importar el complemento Folium DivIcon
from folium.features import DivIcon

Si necesita refrescar su memoria sobre folium, puede descargar y consultar este laboratorio de folium anterior:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Tarea 1: marcar todos los sitios de lanzamiento en un mapa


Primero, intentemos agregar la ubicación de cada sitio en un mapa usando las coordenadas de latitud y longitud del sitio.


El siguiente conjunto de datos con el nombre `spacex_launch_geo.csv` es un conjunto de datos aumentado con latitud y longitud agregadas para cada sitio.


In [9]:
# Descargue y lea `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

100% [................................................................................] 7710 / 7710

Ahora, puedes echar un vistazo a cuáles son las coordenadas de cada sitio.


In [10]:
# Seleccione las subcolumnas relevantes: `Sitio de lanzamiento`, `Lat(Latitud)`, `Long(Longitud)`, `clase`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Las coordenadas anteriores son simplemente números que no pueden brindarle ninguna idea intuitiva sobre dónde están esos sitios de lanzamiento. Si eres muy bueno en geografía, podrás interpretar esos números directamente en tu mente. Si no, también está bien. Visualicemos esas ubicaciones fijándolas en un mapa.


Primero necesitamos crear un objeto "Map" de folium, con una ubicación central inicial que será el Centro Espacial Johnson de la NASA en Houston, Texas.


In [11]:
# La ubicación de inicio es el Centro Espacial Johnson de la NASA.
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

Podríamos usar `folium.Circle` para agregar un área circular resaltada con una etiqueta de texto en una coordenada específica. Por ejemplo,


In [12]:
# Cree un círculo azul en las coordenadas del Centro Espacial Johnson de la NASA con una etiqueta emergente que muestre su nombre.
circle = folium.Circle(nasa_coordinate, radius=1000, color='# d35400', fill=True).add_child(folium.Popup('Centro espacial Johnson de la NASA'))
# Cree un círculo azul en las coordenadas del Centro Espacial Johnson de la NASA con un icono que muestre su nombre.
marker = folium.map.Marker(
    nasa_coordinate,
    # Crear un icono como etiqueta de texto
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:# d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

y deberías encontrar un pequeño círculo amarillo cerca de la ciudad de Houston y puedes acercarte para ver un círculo más grande.


Ahora, agreguemos un círculo para cada sitio de lanzamiento en el marco de datos `launch_sites`.


_TODO:_ Cree y agregue `folium.Circle` y `folium.Marker` para cada sitio de lanzamiento en el mapa del sitio


Un ejemplo de folio.Círculo:


`folium.Circle(coordenada, radio=1000, color='#000000', relleno=True).add_child(folium.Popup(...))`


Un ejemplo de folium.Marker:


`folium.map.Marker(coordinada, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [20]:
# Inicialice el mapa
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# Para cada sitio de lanzamiento, agregue un objeto Círculo según sus valores de coordenadas (Lat, Long). Además, agregue el nombre del sitio de lanzamiento como una etiqueta emergente
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]

    #Añadimos los puntos de los lugares de lanzamiento
    folium.Circle(
        location=coordinate,
        radius=1000,
        color='#000000',
        fill=True,
        fill_color='#3186cc'
    ).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)

    #Añadimos marcador
    folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{row["Launch Site"]}</b></div>'
        )
    ).add_to(site_map)

site_map
        

El mapa generado con los sitios de lanzamiento marcados debería verse similar al siguiente:


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</centro>


Ahora puedes explorar el mapa acercando o alejando las áreas marcadas.
, e intente responder las siguientes preguntas:
- ¿Están todos los sitios de lanzamiento cerca de la línea ecuatorial?
- ¿Están todos los sitios de lanzamiento muy cerca de la costa?

También intente explicar sus hallazgos.


# Tarea 2: marcar los lanzamientos exitosos/fallidos de cada sitio en el mapa


A continuación, intentemos mejorar el mapa agregando los resultados del lanzamiento de cada sitio y veamos qué sitios tienen altas tasas de éxito.
Recuerde que el marco de datos spacex_df tiene registros de lanzamiento detallados y la columna "Class" indica si este lanzamiento fue exitoso o no.


In [21]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


A continuación, creemos marcadores para todos los registros de lanzamiento. 
Si un lanzamiento fue exitoso `(class=1)`, entonces usamos un marcador verde y si un lanzamiento falló, usamos un marcador rojo `(class=0)`


Tenga en cuenta que un lanzamiento solo ocurre en uno de los cuatro sitios de lanzamiento, lo que significa que muchos registros de lanzamiento tendrán exactamente las mismas coordenadas. Los grupos de marcadores pueden ser una buena manera de simplificar un mapa que contiene muchos marcadores que tienen la misma coordenada.


Primero creemos un objeto `MarkerCluster`


In [22]:
marker_cluster = MarkerCluster()


_TODO:_ Cree una nueva columna en el marco de datos `launch_sites` llamada `marker_color` para almacenar los colores del marcador según el valor de `class`


In [23]:
# Función para asignar color al resultado del lanzamiento.

# Aplicar una función para verificar el valor de la columna "clase"
# Si clase = 1, el valor del marcador_color será verde
# Si clase = 0, el valor del marcador_color será rojo
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ Para cada resultado de lanzamiento en el marco de datos `spacex_df`, agregue un `folium.Marker` a `marker_cluster`


In [24]:
# Agregar marcador_cluster al mapa del sitio actual
site_map.add_child(marker_cluster)

# para cada fila en el marco de datos spacex_df
# crear un objeto marcador con su coordenada
# y personalizar la propiedad del icono del Marcador para indicar si este lanzamiento fue exitoso o fallido,
# por ejemplo, icon=folium.Icon(color='blanco', icon_color=fila['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Crear y agregar un grupo de marcadores al mapa del sitio
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color=record['marker_color'], icon='info-sign')
    )
    marker_cluster.add_child(marker)

site_map

Su mapa actualizado puede parecerse a las siguientes capturas de pantalla:


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</centro>


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</centro>


A partir de los marcadores etiquetados con colores en los grupos de marcadores, debería poder identificar fácilmente qué sitios de lanzamiento tienen tasas de éxito relativamente altas.


# TAREA 3: Calcular las distancias entre un sitio de lanzamiento y sus proximidades


A continuación, debemos explorar y analizar las proximidades de los sitios de lanzamiento.


Primero agreguemos una `MousePosition` en el mapa para obtener las coordenadas para pasar el mouse sobre un punto en el mapa. Como tal, mientras explora el mapa, puede encontrar fácilmente las coordenadas de cualquier punto de interés (como el ferrocarril).


In [25]:
# Agregue la posición del mouse para obtener las coordenadas (Lat, Long) al pasar el mouse sobre el mapa.
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Ahora acérquese a un sitio de lanzamiento y explore su proximidad para ver si puede encontrar fácilmente algún ferrocarril, carretera, línea costera, etc. Mueva el mouse a estos puntos y marque sus coordenadas (que se muestran en la parte superior izquierda) en orden a la distancia al sitio de lanzamiento.


Puede calcular la distancia entre dos puntos en el mapa en función de sus valores "Lat" y "Long" utilizando el siguiente método:


In [26]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # radio aproximado de la tierra en km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Marque un punto en la costa más cercana usando MousePosition y calcule la distancia entre el punto de la costa y el lugar de lanzamiento.


In [32]:
# encontrar las coordenadas de la costa más cercana
# por ejemplo: Lat: 28.56367 Long: -80.57163
# distancia_costa = calcular_distancia(lat_sitio_lanzamiento, lon_sitio_lanzamiento, lat_costa, lon_costa)
coastline_lat, coastline_lon = 28.56367, -80.57163 # using mouse point
launch_site_lat, launch_site_lon = 28.573255,	-80.646895	
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

7.42932194658424

_TODO:_ Después de obtener sus coordenadas, cree un `folium.Marker` para mostrar la distancia


In [33]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=folium.DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

site_map.add_child(distance_marker)
lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=2)

_TODO:_ Dibuje una `PolyLine` entre un sitio de lanzamiento y el punto de la costa seleccionado


In [34]:
# Cree un objeto `folium.PolyLine` usando las coordenadas de la costa y las coordenadas del sitio de lanzamiento.
# líneas=folium.PolyLine(ubicaciones=coordenadas, peso=1)
site_map.add_child(lines)

site_map

Su mapa actualizado con línea de distancia debería verse como la siguiente captura de pantalla:


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</centro>


_TODO:_ De manera similar, puedes dibujar una línea entre un sitio de lanzamiento y su ciudad, ferrocarril, carretera, etc. más cercana. Primero debes usar `MousePosition` para encontrar sus coordenadas en el mapa.


Un símbolo en un mapa ferroviario puede verse así:


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</centro>


Un símbolo en un mapa de carreteras puede verse así:


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</centro>


Un símbolo en un mapa de la ciudad puede verse así:


<centro>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</centro>


In [35]:
# Cree un marcador con la distancia a la ciudad, ferrocarril, carretera, etc. más cercana.
# Dibuja una línea entre el marcador y el sitio de lanzamiento.
railway_lat, railway_lon = 28.57591, -80.58586 
highway_lat, highway_lon = 28.57707,	-80.5747	
city_lat, city_lon = 28.07834, -80.60978 

In [36]:
distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

# Railway marker and line
railway_marker = folium.Marker(
    [railway_lat, railway_lon],
    icon=folium.DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railway),
    )
)
site_map.add_child(railway_marker)
railway_line = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]], weight=2)
site_map.add_child(railway_line)

# Highway marker and line
highway_marker = folium.Marker(
    [highway_lat, highway_lon],
    icon=folium.DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway),
    )
)
site_map.add_child(highway_marker)
highway_line = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]], weight=2)
site_map.add_child(highway_line)

# City marker and line
city_marker = folium.Marker(
    [city_lat, city_lon],
    icon=folium.DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city),
    )
)
site_map.add_child(city_marker)
city_line = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [city_lat, city_lon]], weight=2)
site_map.add_child(city_line)

site_map

Después de trazar líneas de distancia hasta las proximidades, podrá responder fácilmente a las siguientes preguntas:
- ¿Están los sitios de lanzamiento muy cerca de las vías férreas?
- ¿Están los sitios de lanzamiento muy cerca de las autopistas?
- ¿Están los sitios de lanzamiento muy cerca de la costa?
- ¿Los sitios de lanzamiento mantienen cierta distancia de las ciudades?

También intente explicar sus hallazgos.


# Próximos pasos:

Ahora ha descubierto muchas ideas interesantes relacionadas con la ubicación de los sitios de lanzamiento utilizando folium, de una manera muy interactiva. A continuación, deberá crear un panel utilizando Ploty Dash con registros de lanzamiento detallados.


## Autores


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Otros contribuyentes


Jose Santarcangelo


## Registro de cambios


|Fecha (AAAA-MM-DD)|Versión|Cambiado por|Cambiar descripción|
|-|-|-|-|
|2021-05-26|1.0|Yan|Creé la versión inicial|


Copyright © 2021 IBM Corporación. Reservados todos los derechos.
